# Session 2, Part 2: dynamic simulation

Run each cell with **Shift+Enter**, from the top. Cells marked **Your turn** are exercises at level **1** (change a value), **2** (a few lines of code), or **3** (advanced, optional). Every exercise cell already runs as is. Each exercise has a hint, and a solution that stays hidden until you click **Show solution**.

Everything so far was **steady state**: one mass balance per unit, solved once (with iteration around recycles). A **dynamic** simulation tracks how the state of each unit, for example concentrations in a reactor, changes over time, driven by differential equations (ODEs) and by inputs that can vary with time. If you have used WEST, GPS-X, BioWin, or SUMO, this is the same kind of simulation: you get time series instead of a single answer.

We use the **Benchmark Simulation Model No. 1 (BSM1)**: a standard activated sludge plant (two anoxic and three aerated reactors with ASM1, and a clarifier), available in EXPOsan.

In [ ]:
# Google Colab only: installs QSDsan the first time. On your own computer this cell does nothing.
# After the install, restart the session (Runtime, Restart session) and run the notebook again from the top.
import sys
if 'google.colab' in sys.modules:
    try:
        import qsdsan
    except ImportError:
        %pip install -q qsdsan==1.5.3 exposan==1.5.3
        print('Installed. Now restart the session (Runtime, Restart session) and run the notebook again from the top.')
    else:
        import biosteam as bst
        bst.preferences.graphviz_format = 'png'   # Colab does not draw the default SVG diagrams

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import qsdsan as qs
print(f'QSDsan {qs.__version__}')

## 1. Run BSM1 dynamically

In [ ]:
from exposan import bsm1

bsm1.load()
sys = bsm1.sys
sys.diagram()

In [ ]:
# Which units run dynamically? Units with ODEs (hasode) carry states; others, such as Mixer and Splitter, are dynamic with algebraic equations only.
[(u.ID, type(u).__name__, u.isdynamic, u.hasode) for u in sys.units]

In [ ]:
# A dynamic system needs a time span. The usual call for a steady-state system does not work:
try:
    sys.simulate()
except Exception as e:
    print(f'{type(e).__name__}: {e}')

A dynamic simulation needs a **time span** `t_span=(start, end)` in days, a **solver** `method` (we will compare them below), and a decision about what to do with the **initial state**. `state_reset_hook='reset_cache'` sets every unit back to its initial conditions before the run.

The initial conditions that come with the BSM1 module are not a steady state: the plant needs about 50 days to settle (with a constant influent). So we first run 100 days.

In [ ]:
eff = sys.flowsheet.stream.effluent       # the treated effluent stream

t0 = time.time()
sys.simulate(t_span=(0, 100), method='BDF', state_reset_hook='reset_cache')
print(f'100 days simulated in {time.time() - t0:.1f} s')

fig, ax = eff.scope.plot_time_series(('S_NH', 'S_NO'))      # ammonium and nitrate in the effluent, mg/L
fig

The effluent ammonium settles at about 1.7 mg N/L after about 50 days, which is the published BSM1 steady-state value. The results are stored in the **scope** of each tracked unit or stream, with `time_series` (the times) and `record` (the values).

Only some objects are tracked, to save memory. `sys.scope.subjects` lists them, and `sys.set_dynamic_tracker(...)` chooses them.

In [ ]:
print('Tracked:', [getattr(s, 'ID', s) for s in sys.scope.subjects])
A1 = sys.flowsheet.unit.A1
fig, ax = A1.scope.plot_time_series(('S_S', 'S_NO'))      # the first (anoxic) reactor
fig

**Your turn 1 (level 1).** Track the last aerated reactor `O3` as well (together with `A1` and the effluent), simulate 20 days again, and plot its dissolved oxygen `S_O` and ammonium `S_NH`. Why is the oxygen in `O3` higher than in `A1`?

<details><summary>Hint</summary>

`sys.set_dynamic_tracker` takes the units and streams to track. Re-run the simulation after changing it, because the records are filled during the run. `A1` is anoxic (no aeration) and `O3` is aerated.

</details>

In [ ]:
O3 = sys.flowsheet.unit.O3
sys.set_dynamic_tracker(A1, eff)           # Your turn: add O3 to the tracked objects
sys.simulate(t_span=(0, 20), method='BDF', state_reset_hook='reset_cache')
fig, ax = A1.scope.plot_time_series(('S_O', 'S_NH'))     # Your turn: plot O3 instead
fig

<details><summary>Show solution</summary>

```python
O3 = sys.flowsheet.unit.O3
sys.set_dynamic_tracker(A1, O3, eff)
sys.simulate(t_span=(0, 20), method='BDF', state_reset_hook='reset_cache')
fig, ax = O3.scope.plot_time_series(('S_O', 'S_NH'))
fig
```

</details>

## 2. Solver settings

`sys.simulate` passes its options to SciPy's ODE solver `solve_ivp`. The ones you use most:

| Option | Meaning |
|---|---|
| `t_span` | start and end time of the simulation, in days |
| `method` | the ODE solver: `'BDF'` and `'LSODA'` (for stiff problems), `'RK23'` and `'RK45'` (explicit, for non-stiff problems), `'Radau'`, and others |
| `rtol`, `atol` | relative and absolute error tolerances of the solver (defaults 1e-3 and 1e-6) |
| `t_eval` | the times at which the states are written when you export them with `export_state_to` (the scope always keeps one record per solver step) |
| `state_reset_hook` | `'reset_cache'`: start from the initial conditions |
| `export_state_to` | a file path, to save the states over time (`.xlsx`, `.csv`, `.tsv`, or `.npy`) |

Wastewater process models are **stiff**: some processes (for example oxygen transfer) are much faster than others (growth of nitrifiers). Solvers made for stiff problems take larger steps and are much faster here.

Options that SciPy does not recognize may be ignored without an error, so check the spelling of `rtol` and `atol`: a typo means the run used the defaults.

In [ ]:
def at(scope, name, day):
    '''Value of a tracked variable at a given time (days), read from a scope.'''
    names = [h[1].split()[0] for h in scope.header]
    return float(np.interp(day, np.asarray(scope.time_series), scope.record[:, names.index(name)]))

sys.set_dynamic_tracker(A1, eff)

def run_case(t_end=5, **kw):
    t0 = time.time()
    sys.simulate(t_span=(0, t_end), state_reset_hook='reset_cache', **kw)
    return time.time() - t0, at(eff.scope, 'S_NH', t_end)

for method in ('BDF', 'LSODA'):
    dt, nh = run_case(method=method)
    print(f'{method:<6}: {dt:5.1f} s, effluent S_NH after 5 days = {nh:.4f} mg/L')

The explicit solvers `'RK23'` and `'RK45'` are much slower on this stiff system (about 10 and 17 seconds in our test, compared with about 2 seconds). You will try them in the exercise below.

**Tolerances** trade accuracy for speed. A looser tolerance is faster but less accurate, a tighter one is more accurate and sometimes much slower.

In [ ]:
for rtol, atol in ((1e-2, 1e-3), (1e-3, 1e-6), (1e-5, 1e-8)):
    dt, nh = run_case(method='BDF', rtol=rtol, atol=atol)
    print(f'rtol {rtol:g}, atol {atol:g}: {dt:5.1f} s, S_NH after 5 days = {nh:.4f} mg/L')

On our machine, a much tighter setting (`rtol=1e-6`, `atol=1e-8`) took 80 seconds for the same 5 days, with no visible change in the result. So choose the loosest tolerance that gives results that do not change when you tighten it.

**Your turn 2 (level 2).** Compare the solvers `'BDF'`, `'LSODA'`, and `'RK23'` for a 5-day simulation: print the run time and the effluent `S_NH` after 5 days for each. Which is fastest? Do the three results agree? (RK23 takes about 10 seconds.)

<details><summary>Hint</summary>

The helper `run_case` takes the solver name as `method`. All three should agree to about three decimals; the time is what differs.

</details>

In [ ]:
for method in ('BDF',):                    # Your turn: add 'LSODA' and 'RK23'
    dt, nh = run_case(method=method)
    print(f'{method:<6}: {dt:5.1f} s, S_NH after 5 days = {nh:.4f} mg/L')

<details><summary>Show solution</summary>

```python
for method in ('BDF', 'LSODA', 'RK23'):
    dt, nh = run_case(method=method)
    print(f'{method:<6}: {dt:5.1f} s, S_NH after 5 days = {nh:.4f} mg/L')
```

</details>

## 3. Time series as inputs

Real plants have changing loads and conditions. QSDsan has two tools for time-dependent inputs.

### 3.1. `ExogenousDynamicVariable`

A variable that changes with time but does not depend on any unit, such as ambient temperature or sunlight. Define it as a function of time or from data, then read it with `variable(t)`. Units that accept such variables (the `exogenous_vars` argument) read them during the simulation. The EXPOsan `metab` and `pm2_batch` modules use them for temperature and light.

In [ ]:
from qsdsan.utils import ExogenousDynamicVariable as EDV

# From a function of time (days): temperature in K that swings by 5 degrees every day
T = EDV('T', function=lambda t: 298.15 + 5 * np.sin(2 * np.pi * t))

# From data points (time series), for example a logged signal
t_data = np.linspace(0, 5, 51)
T_logged = EDV('T_logged', t=t_data, y=298.15 + 5 * np.sin(t_data))

print(f'T(0.25 d) = {T(0.25):.2f} K, T_logged(1.0 d) = {T_logged(1.0):.2f} K')

### 3.2. `DynamicInfluent`

A unit that produces a wastewater stream whose flow and concentrations follow a time series. With no arguments it loads the BSM1 dry-weather influent that ships with QSDsan: 14 days of data, with the concentrations in mg/L and the flow `Q` in m3/d. Time is in days.

In [ ]:
from qsdsan.unit_operations import DynamicInfluent
from qsdsan.unit_operations.dynamic._influent import dynamic_inf_path

inf_data = pd.read_csv(dynamic_inf_path, sep='\t')
print(f'{len(inf_data)} rows, columns: {list(inf_data.columns)}')
inf_data.head(3).round(2)

In [ ]:
di = DynamicInfluent('DI', outs=('dynamic_stream',))
di_sys = qs.System('di_sys', path=(di,))
di_sys.set_dynamic_tracker(di)
di_sys.simulate(t_span=(0, 14), max_step=0.05)    # adding a step limit for finer recording

fig, ax = di.scope.plot_time_series(('S_NH', 'S_S'))
fig

**Your turn 3 (level 2).** Make your own influent: take the data above, double the ammonium (`S_NH`) column, save it as a CSV file, and give the file to a new `DynamicInfluent` through `data_file=`. Simulate 14 days, plot `S_NH`, and report the peak ammonium concentration of the original and of your influent.

<details><summary>Hint</summary>

A pandas column can be scaled in place, for example `my_data['S_NH'] *= 2`. The influent unit reads the file when it is created.

</details>

In [ ]:
import os, tempfile

my_data = inf_data.copy()
# Your turn: change the S_NH column of my_data
path = os.path.join(tempfile.mkdtemp(), 'my_influent.csv')
my_data.to_csv(path, index=False)

di2 = DynamicInfluent('DI2', outs=('my_stream',), data_file=path)
sys2 = qs.System('di_sys2', path=(di2,))
sys2.set_dynamic_tracker(di2)
sys2.simulate(t_span=(0, 14), max_step=0.05)
fig, ax = di2.scope.plot_time_series(('S_NH',))
fig

<details><summary>Show solution</summary>

```python
import os, tempfile

my_data = inf_data.copy()
my_data['S_NH'] *= 2
path = os.path.join(tempfile.mkdtemp(), 'my_influent.csv')
my_data.to_csv(path, index=False)

di2 = DynamicInfluent('DI2', outs=('my_stream',), data_file=path)
sys2 = qs.System('di_sys2', path=(di2,))
sys2.set_dynamic_tracker(di2)
sys2.simulate(t_span=(0, 14), max_step=0.05)
fig, ax = di2.scope.plot_time_series(('S_NH',))
print(f'peak S_NH: original {inf_data["S_NH"].max():.1f} mg/L, yours {my_data["S_NH"].max():.1f} mg/L')
fig
```

</details>

In a full model, the influent unit feeds the first reactor, so the plant sees the time-varying load (the EXPOsan `pm2_ecorecover` module does this). Setting that up for a plant of your own is the main step beyond this part, and the pattern is the one in the tutorial.

## 4. Control and hybrid modeling

You asked about control and hybrid modeling. The honest summary of where QSDsan stands:

- **Control.** There is no formal controller module. A controller can be written as part of a custom dynamic unit (a `SanUnit` with its own `_compile_ODE` or `_compile_AE`, as in section 2 of tutorial 11) that reads a state and changes an input or setpoint. This is possible but you write it yourself, and it is not yet a documented workflow.
- **Hybrid modeling** (for example a mechanistic model combined with a data-driven model). There is no formal mechanism for it. If you build one, please share it in the community channel, because several people asked.

Both are good topics to discuss with the community, and contributions are welcome.

## Practical advice

1. **Let the plant settle.** The initial conditions are not a steady state, so simulate long enough (about 50 days for BSM1) and read the results after that.
2. **Use a stiff solver** (`'BDF'` or `'LSODA'`) for biological process models, and check that tightening the tolerance does not change your results.
3. **Track only what you need** with `set_dynamic_tracker`, because every tracked unit stores its full time series.
4. **Time a short run first**, then scale up. Long runs of larger systems (for example `bsm2`, `werf`) can take many minutes.
5. **Save time series** with `export_state_to` or by writing the scope records to a file.

## Where to go next

- Tutorial: [Dynamic Simulation](https://qsdsan.readthedocs.io/en/latest/tutorials/11_Dynamic_Simulation.html), which also shows how to write your own dynamic unit, and [Modeling notes and pitfalls](https://qsdsan.readthedocs.io/en/latest/tutorials/14_Modeling_Notes_and_Pitfalls.html) for dynamic simulation pitfalls.
- EXPOsan modules to read and reuse: `bsm1`, `bsm2`, `metab`, `pm2_batch`, `pm2_ecorecover`.
- [SciPy's `solve_ivp`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html) documents all solver options.